# Phase 2: Data Cleaning and Preprocessing

## Objective

The objective of this phase is to clean and preprocess the raw book dataset collected during the web-scraping phase.

The cleaning process includes:
- Inspecting the dataset structure
- Checking duplicate records
- Handling missing values
- Converting numerical fields into appropriate data types
- Standardizing categorical and text fields
- Validating the cleaned data
- Creating useful features for analysis
- Saving the final cleaned dataset

## 1. Loading the Raw Dataset

The raw dataset generated during Phase 1 is loaded into a Pandas DataFrame.

The number of rows and columns is checked to confirm that the dataset has been loaded successfully.

In [1]:
import pandas as pd

df = pd.read_csv("raw_books_data.csv")

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded successfully!
Rows: 1000
Columns: 10


## 2. Inspecting Dataset Structure

The `info()` method is used to inspect the dataset structure, including column names, data types, and non-null values.

This provides an initial understanding of the data before performing cleaning operations.

In [2]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   title         1000 non-null   str  
 1   price         1000 non-null   str  
 2   rating        1000 non-null   str  
 3   availability  1000 non-null   str  
 4   product_url   1000 non-null   str  
 5   category      1000 non-null   str  
 6   upc           1000 non-null   str  
 7   stock_count   1000 non-null   int64
 8   tax           1000 non-null   str  
 9   description   998 non-null    str  
dtypes: int64(1), str(9)
memory usage: 78.3 KB
None


## 3. Checking for Duplicate Records

Duplicate rows are identified using Pandas' `duplicated()` method.

Removing duplicate records is important to prevent repeated observations from affecting subsequent analysis.

In [3]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


## 4. Checking Missing Values

Missing values are identified for every column using `isnull().sum()`.

The results are used to determine which fields require missing-value treatment.

In [4]:
missing_values = df.isnull().sum()

print(missing_values)

title           0
price           0
rating          0
availability    0
product_url     0
category        0
upc             0
stock_count     0
tax             0
description     2
dtype: int64


## 5. Handling Missing Descriptions

Missing values in the `description` column are replaced with the text **"No description available"**.

This ensures that missing descriptions are handled consistently while retaining all book records.

In [5]:
df["description"] = df["description"].fillna("No description available")

print("Missing descriptions after cleaning:", df["description"].isnull().sum())

Missing descriptions after cleaning: 0


## 6. Cleaning the Price Column

The `price` column initially contains currency symbols and text formatting.

The currency symbols are removed, whitespace is stripped, and the values are converted to the `float` data type so that prices can be used for numerical analysis.

In [6]:
df["price"] = (
    df["price"]
    .str.replace("Â£", "", regex=False)
    .str.replace("£", "", regex=False)
    .str.strip()
    .astype(float)
)

print(df["price"].dtype)
print(df["price"].head())

float64
0    51.77
1    53.74
2    50.10
3    47.82
4    54.23
Name: price, dtype: float64


## 7. Converting Book Ratings to Numeric Values

The original ratings are represented as words such as `One`, `Two`, `Three`, `Four`, and `Five`.

These values are mapped to numerical ratings from **1 to 5** to make the rating variable suitable for statistical analysis and machine learning.

In [7]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["rating"] = df["rating"].map(rating_map)

print(df["rating"].dtype)
print(df["rating"].value_counts().sort_index())

int64
rating
1    226
2    196
3    203
4    179
5    196
Name: count, dtype: int64


## 8. Cleaning Tax and Availability Data

The `tax` column is cleaned by removing currency symbols and converting the values to a numeric data type.

The availability information is standardized into consistent **In Stock** values.

In [8]:
df["tax"] = (
    df["tax"]
    .str.replace("Â£", "", regex=False)
    .str.replace("£", "", regex=False)
    .str.strip()
    .astype(float)
)

print(df["tax"].dtype)
print(df["tax"].head())

float64
0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: tax, dtype: float64


In [9]:
id="q8x2lm"
df["availability"] = df["availability"].str.contains(
    "In stock", case=False, na=False
)

df["availability"] = df["availability"].map({
    True: "In Stock",
    False: "Out of Stock"
})

print(df["availability"].value_counts())

availability
In Stock    1000
Name: count, dtype: int64


## 9. Reviewing Data Types

The data types of all columns are reviewed after the initial cleaning operations.

This confirms that numerical variables have appropriate numeric data types and that text fields remain suitable for further processing.

In [10]:
print(df.dtypes)

title               str
price           float64
rating            int64
availability        str
product_url         str
category            str
upc                 str
stock_count       int64
tax             float64
description         str
dtype: object


## 10. Validating Numerical Values

The dataset is checked for invalid numerical values, including:

- Prices below zero
- Ratings outside the 1–5 range
- Stock counts below zero

These checks help confirm that the numerical fields contain valid values.

In [11]:
print("Price below 0:", (df["price"] < 0).sum())
print("Rating outside 1-5:", ((df["rating"] < 1) | (df["rating"] > 5)).sum())
print("Stock below 0:", (df["stock_count"] < 0).sum())
print("Tax below 0:", (df["tax"] < 0).sum())

Price below 0: 0
Rating outside 1-5: 0
Stock below 0: 0
Tax below 0: 0


## 11. Checking Categorical Values

The unique availability values, number of book categories, and number of unique ratings are examined.

This helps verify the consistency of the categorical variables before feature engineering.

In [12]:
print("Unique availability values:")
print(df["availability"].unique())

print("\nNumber of categories:")
print(df["category"].nunique())

print("\nNumber of unique ratings:")
print(df["rating"].nunique())

Unique availability values:
<StringArray>
['In Stock']
Length: 1, dtype: str

Number of categories:
50

Number of unique ratings:
5


## 12. Standardizing Text Fields

Leading and trailing whitespace is removed from the main text and categorical columns.

This standardization helps maintain consistent values and prevents formatting differences from affecting later analysis.

In [13]:
text_columns = [
    "title",
    "availability",
    "product_url",
    "category",
    "upc",
    "description"
]

for column in text_columns:
    df[column] = df[column].str.strip()

print("Text fields cleaned successfully.")

Text fields cleaned successfully.


## 13. Creating Price Categories

A new `price_category` feature is created to group books according to their price:

- **Budget** – price below £20
- **Mid-Range** – price from £20 to below £40
- **Premium** – price of £40 or above

This feature makes it easier to compare books across different price segments.

In [14]:
def categorize_price(price):
    if price < 20:
        return "Budget"
    elif price < 40:
        return "Mid-Range"
    else:
        return "Premium"

df["price_category"] = df["price"].apply(categorize_price)

print(df["price_category"].value_counts())

price_category
Premium      403
Mid-Range    401
Budget       196
Name: count, dtype: int64


## 14. Creating Rating Categories

A new `rating_category` feature is created to group books according to their numerical rating:

- **Low Rated** – rating 1–2
- **Average Rated** – rating 3
- **Highly Rated** – rating 4–5

This categorical feature is useful for comparing groups of books during EDA and dashboard analysis.

In [15]:
def categorize_rating(rating):
    if rating <= 2:
        return "Low Rated"
    elif rating == 3:
        return "Average Rated"
    else:
        return "Highly Rated"

df["rating_category"] = df["rating"].apply(categorize_rating)

print(df["rating_category"].value_counts())

rating_category
Low Rated        422
Highly Rated     375
Average Rated    203
Name: count, dtype: int64


## 15. Feature Engineering: Description Length

A new `description_length` feature is created by calculating the number of characters in each book description.

This provides a numerical representation of the amount of descriptive text associated with each book.

In [16]:
df["description_length"] = df["description"].str.len()

print(df["description_length"].head())
print("\nAverage description length:", df["description_length"].mean())

0    1017
1    1029
2    1136
3    1647
4    1969
Name: description_length, dtype: int64

Average description length: 1441.497


## 16. Final Data Quality Check

The cleaned dataset is checked again for:

- Missing values
- Duplicate records
- Final number of rows and columns

This confirms that the main cleaning and preprocessing operations have been completed successfully.

In [17]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nFinal shape:", df.shape)

Missing values:
title                 0
price                 0
rating                0
availability          0
product_url           0
category              0
upc                   0
stock_count           0
tax                   0
description           0
price_category        0
rating_category       0
description_length    0
dtype: int64

Duplicate rows: 0

Final shape: (1000, 13)


## 17. Saving the Cleaned Dataset

The final cleaned and feature-engineered dataset is saved as:

`cleaned_books_data.csv`

This file is used as the input dataset for the next phase, **Exploratory Data Analysis (EDA)**.

In [18]:
df.to_csv("cleaned_books_data.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


# Phase 2 Conclusion

The raw book dataset has been successfully cleaned and preprocessed.

The process included handling missing values, checking duplicates, converting data types, standardizing text and categorical fields, validating numerical values, and creating the `price_category`, `rating_category`, and `description_length` features.

The final cleaned dataset is saved as `cleaned_books_data.csv` and is ready for **Phase 3: Exploratory Data Analysis (EDA)**.